# Import libraries

In [ ]:
from utils.path_helpers import repo_path
from Simulation.system_functions import PolymerCSTR
import numpy as np
import os
from utils.td3_helpers import load_and_prepare_system_data, filling_the_buffer, add_steady_state_samples, print_accuracy, ReplayDataset
import torch
from TD3Agent.agent import TD3Agent
from Simulation.mpc import MpcSolver
from torch.utils.data import DataLoader

## Initialize the system

In [ ]:
# First initiate the system
# Parameters
Ad = 2.142e17           # h^-1
Ed = 14897              # K
Ap = 3.816e10           # L/(molh)
Ep = 3557               # K
At = 4.50e12            # L/(molh)
Et = 843                # K
fi = 0.6                # Coefficient
m_delta_H_r = -6.99e4   # j/mol
hA = 1.05e6             # j/(Kh)
rhocp = 1506            # j/(Kh)
rhoccpc = 4043          # j/(Kh)
Mm = 104.14             # g/mol
system_params = np.array([Ad, Ed, Ap, Ep, At, Et, fi, m_delta_H_r, hA, rhocp, rhoccpc, Mm])

In [ ]:
# Design Parameters
CIf = 0.5888    # mol/L
CMf = 8.6981    # mol/L
Qi = 108.       # L/h
Qs = 459.       # L/h
Tf = 330.       # K
Tcf = 295.      # K
V = 3000.       # L
Vc = 3312.4     # L

system_design_params = np.array([CIf, CMf, Qi, Qs, Tf, Tcf, V, Vc])

In [ ]:
# Steady State Inputs
Qm_ss = 378.    # L/h
Qc_ss = 471.6   # L/h

system_steady_state_inputs = np.array([Qc_ss, Qm_ss])

In [ ]:
# Sampling time of the system
delta_t = 0.5 # 30 mins

In [ ]:
# Initiate the CSTR for steady state values
cstr = PolymerCSTR(system_params, system_design_params, system_steady_state_inputs, delta_t)
steady_states={"ss_inputs":cstr.ss_inputs,
               "y_ss":cstr.y_ss}

## Loading the system matrices, min max scaling, and min max of the states

In [ ]:
dir_path = os.path.join(os.fspath(repo_path()), "Data")

In [ ]:
# Defining the range of setpoints for data generation
setpoint_y = np.array([[2.8, 320.],
                       [5., 326.]])
u_min = np.array([71.6, 78])
u_max = np.array([870, 670])

system_data = load_and_prepare_system_data(steady_states=steady_states, setpoint_y=setpoint_y, u_min=u_min, u_max=u_max)

In [ ]:
A_aug = system_data["A_aug"]
B_aug = system_data["B_aug"]
C_aug = system_data["C_aug"]

In [ ]:
data_min = system_data["data_min"]
data_max = system_data["data_max"]

In [ ]:
min_max_states = {'max_s': np.array([256.79686253, 256.01560603,  48.99447186, 144.79949103,
          2.82199733,   3.14014989,   2.78866348,   3.71691422,
          6.2029936 ]),
                  'min_s': np.array([ -272.28060121, -1112.33972595,   -76.63993491,  -608.60327886,
           -3.94399122,    -3.93115257,    -2.9532091 ,    -4.06547624,
          -28.25906582])}

In [ ]:
y_sp_scaled_deviation = system_data["y_sp_scaled_deviation"]

In [ ]:
b_min = system_data["b_min"]
b_max = system_data["b_max"]

In [ ]:
min_max_dict = system_data["min_max_dict"]
min_max_dict["x_max"] = np.array([256.79686253, 256.01560603,  48.99447186, 144.79949103,
          2.82199733,   3.14014989,   2.78866348,   3.71691422,
          6.2029936 ])
min_max_dict["x_min"] = np.array([ -272.28060121, -1112.33972595,   -76.63993491,  -608.60327886,
           -3.94399122,    -3.93115257,    -2.9532091 ,    -4.06547624,
          -28.25906582])

## Setting The hyperparameters for the TD3 Agent

In [ ]:
set_points_number = int(C_aug.shape[0])
inputs_number = int(B_aug.shape[1])
STATE_DIM = int(A_aug.shape[0]) + set_points_number + inputs_number
ACTION_DIM = int(B_aug.shape[1])
n_outputs = C_aug.shape[0]
ACTOR_LAYER_SIZES = [256] * 3
CRITIC_LAYER_SIZES = [256] * 3
BUFFER_CAPACITY = 5_000_000
ACTOR_LR = 1e-3
CRITIC_LR = 1e-3
SMOOTHING_STD = 0.000001
NOISE_CLIP = 0.000001
GAMMA = 0.99
TAU = 0.005 # 0.01
MAX_ACTION = 1
POLICY_DELAY = 2
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
BATCH_SIZE = 128
STD_START = 0.02
STD_END = 0.001
STD_DECAY_RATE = 0.99992
STD_DECAY_MODE = "exp"

In [ ]:
td3_agent = TD3Agent(
    state_dim=STATE_DIM,
    action_dim=ACTION_DIM,
    actor_hidden=ACTOR_LAYER_SIZES,
    critic_hidden=CRITIC_LAYER_SIZES,
    gamma=GAMMA,
    actor_lr=ACTOR_LR,
    critic_lr=CRITIC_LR,
    batch_size=BATCH_SIZE,
    policy_delay=POLICY_DELAY,
    target_policy_smoothing_noise_std=SMOOTHING_STD,
    noise_clip=NOISE_CLIP,
    max_action=MAX_ACTION,
    tau=TAU,
    std_start=STD_START,
    std_end=STD_END,
    std_decay_rate=STD_DECAY_RATE,
    std_decay_mode=STD_DECAY_MODE,
    buffer_size=BUFFER_CAPACITY,
    device=DEVICE,
    mode="mpc"
    )

# Filling the buffer

In [ ]:
# MPC parameters
predict_h = 9
cont_h = 3
b1 = (b_min[0], b_max[0])
b2 = (b_min[1], b_max[1])
bnds = (b1, b2)*cont_h
cons = []
IC_opt = np.zeros(inputs_number*cont_h)
Q1 = 5
Q2 = 1
R1 = 1
R2 = 1
Q_rew = np.array([[5, 0], [0, 1]])
R_rew = np.array([[1, 0], [0, 1]])

In [ ]:
MPC_obj = MpcSolver(A_aug, B_aug, C_aug,
    Q_out=np.array([Q1, Q2]),
    R_in=np.array([R1, R2]),
    NP=predict_h,
    NC=cont_h)

In [ ]:
steady_states_samples_number = 100000
mpc_pretrain_samples_numbers = BUFFER_CAPACITY - steady_states_samples_number

In [ ]:
filling_the_buffer(
        min_max_dict,
        A_aug, B_aug, C_aug,
        MPC_obj,
        mpc_pretrain_samples_numbers,
        Q_rew, R_rew,
        td3_agent,
        IC_opt, bnds, cons, chunk_size= 100_000)

In [ ]:
add_steady_state_samples(
        min_max_dict,
        A_aug, B_aug, C_aug,
        MPC_obj,
        steady_states_samples_number,
        Q_rew, R_rew,
        td3_agent,
        IC_opt, bnds, cons, chunk_size= 100_000)

## Pre training the Agent

In [ ]:
n = len(td3_agent.buffer)
s = torch.from_numpy(td3_agent.buffer.states[:n]).float()
a = torch.from_numpy(td3_agent.buffer.actions[:n]).float()
r = torch.from_numpy(td3_agent.buffer.rewards[:n]).float()
ns = torch.from_numpy(td3_agent.buffer.next_states[:n]).float()
d = torch.from_numpy(td3_agent.buffer.dones[:n]).float()

dataset = ReplayDataset(s, a, r, ns, d)
pin = DEVICE.type == "cuda"
data_loader = DataLoader(dataset, batch_size=8192, shuffle=True, num_workers=0, pin_memory=pin, drop_last=True)

In [ ]:
td3_agent.pretrain_from_buffer(
    num_actor_epochs=1000,
    num_critic_epochs=500,
    data_loader=data_loader,
    use_target_noise_critic=True,
    log_interval=10,
)

## Saving and loading the agent to make sure the agent has been stored

In [ ]:
filename_agent = td3_agent.save(dir_path)

## Checking the accuracy of the agent and compare it to the MPC actions

In [ ]:
print_accuracy(td3_agent, n_samples=20, device=DEVICE)